# 3. Train and track Random Forest

Train one approach and record its result in the same MLflow experiment. Airflow monitors this notebook as its own task.

## Execution parameters
Airflow injects these values into the tagged parameter cell. For manual execution, keep the defaults and run the five notebooks in order.

In [ ]:
# Notebook inputs.
# Airflow supplies the paths and run ID; manual execution uses these defaults.
project_root = ""
run_dir = ""
workflow_run_id = ""
experiment_name = "iris-pipeline-comparison"
registered_model_name = "iris-pipeline-classifier"

## Locate files and connect to MLflow
SQLite holds experiment and registry records. This run folder holds data and small result files shared between notebook tasks.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

# Find the project folder.
# Looking for requirements.txt lets this notebook start from a subfolder too.
if project_root:
    project_path = Path(project_root).resolve()
else:
    project_path = Path.cwd().resolve()
while not (project_path / "requirements.txt").is_file():
    if project_path == project_path.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_path = project_path.parent

# Choose the output folder.
# Separate Airflow run folders prevent one execution from overwriting another.
if run_dir:
    run_path = Path(run_dir)
else:
    run_path = project_path / "outputs" / "manual"
run_path.mkdir(parents=True, exist_ok=True)

# Connect to MLflow.
# One database keeps both models, their runs, and registered versions together.
tracking_uri = "sqlite:///" + (project_path / "mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()

print("Run folder:", run_path)

## Read the shared split
Each notebook has its own Python session, so it reads the data saved by the first notebook.

In [ ]:
split = json.loads((run_path / "split.json").read_text(encoding="utf-8"))
X_train = pd.DataFrame(
    split["X_train"],
    columns=split["feature_names"],
    dtype="float64",
)
X_test = pd.DataFrame(split["X_test"], columns=split["feature_names"], dtype="float64")
y_train = split["y_train"]
y_test = split["y_test"]
mlflow.set_experiment(experiment_name)

## Train and log progress
We train Random Forest with warm_start=True, retaining existing trees and adding 10 more at each checkpoint until there are 100. Steps are tree counts, not epochs. Log loss evaluates predicted probabilities; it is not the tree-building criterion.

After every checkpoint, calculate training/test accuracy and log loss and send them to MLflow with a step number. Print the same measurements for Airflow task logs. Training time accumulates fitting time only.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, log_loss, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

# Prepare Random Forest for checkpoint training.
# warm_start keeps existing trees when the next fit adds more trees.
model = RandomForestClassifier(warm_start=True, random_state=split["random_state"])
total_trees = 100
trees_per_checkpoint = 10
training_seconds = 0.0

### Run training and tracking
Keep training and logging in one run so a notebook exception marks the run as failed.

In [ ]:
with mlflow.start_run(run_name="random-forest") as run:

    # Record the training settings.
    # MLflow parameters explain which configuration produced this run.
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": total_trees,
        "trees_per_checkpoint": trees_per_checkpoint,
        "warm_start": model.warm_start,
        "max_depth": model.max_depth,
        "random_state": split["random_state"],
        "test_size": split["test_size"],
        "dataset": "iris",
    })

    # Link the model run to Airflow.
    # This tag contains the workflow execution ID shown in the Airflow UI.
    mlflow.set_tag("workflow_run_id", split["execution_id"])
    mlflow.set_tag("progress_unit", "trees")

    for step in range(trees_per_checkpoint, total_trees + 1, trees_per_checkpoint):
        model.set_params(n_estimators=step)
        started = perf_counter()
        model.fit(X_train, y_train)
        training_seconds += perf_counter() - started

        # Evaluate the current model.
        # Training and held-out scores show how its predictions change as training
        # progresses.
        train_predictions = model.predict(X_train)
        predictions = model.predict(X_test)
        train_accuracy = accuracy_score(y_train, train_predictions)
        test_accuracy = accuracy_score(y_test, predictions)
        train_probabilities = model.predict_proba(X_train)
        test_probabilities = model.predict_proba(X_test)

        train_log_loss = log_loss(y_train, train_probabilities, labels=model.classes_)
        test_log_loss = log_loss(y_test, test_probabilities, labels=model.classes_)

        # Track this checkpoint.
        # Reusing metric names with a new step lets MLflow build training curves.
        metrics = {
            "train_accuracy": train_accuracy,
            "test_accuracy": test_accuracy,
            "train_log_loss": train_log_loss,
            "test_log_loss": test_log_loss,
            "training_seconds": training_seconds,
            "accuracy": test_accuracy,
        }
        mlflow.log_metrics(metrics, step=step, synchronous=True)
        print(
            f"Step {step}: train_loss={train_log_loss:.4f}, test_loss={test_log_loss:.4f}, "
            f"train_accuracy={train_accuracy:.3f}, test_accuracy={test_accuracy:.3f}",
            flush=True,
        )

    # Save the final evaluation.
    # This JSON artifact summarizes the last checkpoint; Metrics retains the full
    # history.
    mlflow.log_dict({
        **metrics,
        "final_step": step,
        "train_rows": len(y_train),
        "test_rows": len(y_test),
    }, "evaluation.json")

    # Save the test confusion matrix.
    # Rows show actual classes and columns show predictions; off-diagonal counts are
    # mistakes.
    plot = ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        labels=model.classes_,
        display_labels=load_iris().target_names,
        cmap="Blues",
        colorbar=False,
    )
    plot.ax_.set_title("Random Forest - test predictions")
    plot.figure_.tight_layout()
    mlflow.log_figure(plot.figure_, "confusion_matrix.png")
    plt.show()
    plt.close(plot.figure_)

    # Save a reloadable model.
    # The signature describes its inputs and outputs; the example supplies real input
    # rows.
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        # Allow serialization of our fitted trees.
        # This MLflow version requires the tree type to be explicitly trusted.
        skops_trusted_types=["sklearn.tree._tree.Tree"],
        signature=infer_signature(X_test, predictions),
        input_example=X_train.head(3),
    )

### Review the final results
The last checkpoint supplies the scores used for model comparison.

In [ ]:
print("Run ID:", run.info.run_id)
print("Training accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)
print("Training log loss:", train_log_loss)
print("Test log loss:", test_log_loss)
print("Training seconds:", training_seconds)

## Save the handoff
The comparison notebook needs the run ID and saved model location. A small JSON file carries those references.

In [ ]:
# Pass references to the comparison notebook.
# The next task reads this JSON instead of sharing a Python model across notebook
# kernels.
result = {
    "execution_id": split["execution_id"],
    "model_type": "RandomForestClassifier",
    "run_id": run.info.run_id,
    "model_uri": model_info.model_uri,
}
(run_path / "forest_result.json").write_text(
    json.dumps(result, indent=2),
    encoding="utf-8",
)
print("Saved result:", run_path / "forest_result.json")

## Inspect progress in MLflow
Open Metrics and select train_log_loss and test_log_loss to view their histories against Step. Repeat for train_accuracy and test_accuracy. Logistic Regression steps are epochs; Random Forest steps are tree counts. The latest values are used for model comparison.

These histories are numeric MLflow records, not images. Artifacts contains the final confusion_matrix.png and evaluation.json. Airflow task logs contain a printed line per checkpoint.

Iris trains quickly, so the run may finish before you refresh the UI. Its full history remains available afterward. We do not tune settings using the repeatedly measured test scores; these checkpoints demonstrate tracking.